In [78]:

"""
This notebook provides the computer-assisted sign checks used in Appendix A
(of the paper "Higher-order estimates of a highest wave of the Whitham equation").

The proof fixes a=0.72 and c=0.5 and partitions s in [1/2,1) into subintervals
[s_i,s_{i+1}] of length step = 10^{-3}. On each subinterval the analytic
monotonicity estimates from the text reduce the required inequalities to endpoint
expressions in s_i and s_{i+1}.

The current proof has two sigma regimes. On (1,2] the code checks
B(s_{i+1},s_{i+1}) - 5*overline b_{s_i}(a) > 0. On [2,infinity) it writes
rho=1/sigma and verifies the concavity argument for F_s(rho) on [0,1/2]:
  * F_s''(1/2) < 0, which gives concavity on [0,1/2], uniformly in s;
  * F_s(0) > 0;
  * F_s(1/2) > 0.
Together the endpoint checks give F_s(rho)>0 on [0,1/2].

The code uses interval arithmetic (mpmath.iv) with mp.dps = 200. Each computed
value is a certified enclosure [L_i,U_i] of the corresponding endpoint bound on
[s_i,s_{i+1}]. For positivity checks the reported worst margin is min_i L_i;
for the negativity check F_s''(1/2)<0 it is max_i U_i.

Implementation notes:
  * Beta is evaluated via Gamma(s)^2/Gamma(2s).
  * atanh(c) is computed via 0.5*log((1+c)/(1-c)). Thus, when c=0.5,
    2*atanh(c)=log(3).
  * The endpoint checks for F_s(0) and F_s(1/2) use the interval lower bound
    c^(2s_{i+1})/s_{i+1} - 2*c^(s_i+1)/(s_i+1) - c^(s_i+3)/(s_i+3)
    - 2*atanh(c) + (2/3)*c^3 + 2*c for underline b_s(c). For c=0.5 this is
    2^{-2s_{i+1}}/s_{i+1} - 2^{-s_i}/(s_i+1) - 2^{-s_i-3}/(s_i+3)
    - log(3) + 13/12.
  * The F_s''(1/2) negativity cell is the simplified c=0.5 expression appearing
    in the proof.
"""



'\nThis notebook provides the computer-assisted sign checks used in Appendix A\n(of the paper "Higher-order estimates of a highest wave of the Whitham equation").\n\nThe proof fixes a=0.72 and c=0.5 and partitions s in [1/2,1) into subintervals\n[s_i,s_{i+1}] of length step = 10^{-3}. On each subinterval the analytic\nmonotonicity estimates from the text reduce the required inequalities to endpoint\nexpressions in s_i and s_{i+1}.\n\nThe current proof has two sigma regimes. On (1,2] the code checks\nB(s_{i+1},s_{i+1}) - 5*overline b_{s_i}(a) > 0. On [2,infinity) it writes\nrho=1/sigma and verifies the concavity argument for F_s(rho) on [0,1/2]:\n  * F_s\'\'(1/2) < 0, which gives concavity on [0,1/2], uniformly in s;\n  * F_s(0) > 0;\n  * F_s(1/2) > 0.\nTogether the endpoint checks give F_s(rho)>0 on [0,1/2].\n\nThe code uses interval arithmetic (mpmath.iv) with mp.dps = 200. Each computed\nvalue is a certified enclosure [L_i,U_i] of the corresponding endpoint bound on\n[s_i,s_{i+1}]. F

In [79]:
from mpmath import mp, iv
import numpy as np

In [80]:
mp.dps = 200
a = iv.mpf('0.72')
c = iv.mpf('0.5')

In [81]:
def iv_atanh(x):
    return iv.mpf('0.5') * iv.log((iv.mpf('1') + x) / (iv.mpf('1') - x))

def iv_pow(base,exp):
    return iv.mpf(str(base))**iv.mpf(str(exp))

def beta_iv(s):
    return iv.gamma(s)*iv.gamma(s)/iv.gamma(2*s)

def b_s_upper(s, a=0.72):
    a      = iv.mpf(a)
    s    = iv.mpf(s)      # example

    b_s_upper = (
            (iv_pow(a, iv.mpf('2') * s) / s)
            - (iv.mpf('2') * iv_pow(a, s + iv.mpf('1')) / (s + iv.mpf('1')))
        )
        
    return b_s_upper

def b_s_lower_bound(s_i,s_ip1, c=0.5):
    c      = iv.mpf(c)
    s_i    = iv.mpf(s_i)      # example
    s_ip1  = iv.mpf(s_ip1)

    b_s_lower_bound = (
            (iv_pow(c, iv.mpf('2') * s_ip1) / s_ip1)
            - (iv.mpf('2') * iv_pow(c, s_i + iv.mpf('1')) / (s_i + iv.mpf('1')))
            - (iv_pow(c, s_i + iv.mpf('3')) / (s_i + iv.mpf('3')))
            - (iv.mpf('2') * iv_atanh(c))
            + ((iv.mpf('2') / iv.mpf('3')) * iv_pow(c, iv.mpf('3')))
            + (iv.mpf('2') * c)
        )
        
    return b_s_lower_bound

In [82]:
step=0.001
s_i_list = np.arange(0.5,1,step)

def check_1(s_i_list, step):
    # \sigma \in (1,2]
    # Check for B(s,s)-5b_s>0.
    
    interval_list = []
    worst = None
    for s_i in s_i_list:
        S_i = iv.mpf(s_i)
        S_ip1 = iv.mpf(s_i+step)
        val = beta_iv(S_ip1)-iv.mpf(5)*b_s_upper(S_i)
        interval_list.append(val)
        
        L = val.a
        if worst is None or L<worst:
            worst=L
        
    return interval_list, worst

In [83]:
check_1_result, check_1_smallest = check_1(s_i_list,step)

In [84]:
print(check_1_smallest)

[0.005834578701250592303, 0.005834578701250592303]


In [85]:
# All are >0, so the proof holds.

In [86]:
def check_F_second_at_half_negative(s_i_list, step):
    # Check for F_s''(1/2)<0.
    
    interval_list = []
    worst = None
    
    for s_i in s_i_list:
        S_i = iv.mpf(s_i)
        S_ip1 = iv.mpf(s_i+step)
        val = (
            (iv_pow(iv.mpf(2), iv.mpf(1)-iv.mpf(2)*S_i)-iv_pow(iv.mpf(2), iv.mpf(3)-iv.mpf(1)/S_i))/S_ip1
            +(iv_pow(iv.mpf(2), iv.mpf(4)-iv.mpf(1)/S_ip1)-iv_pow(iv.mpf(2), iv.mpf(1)-S_ip1))/(S_i+iv.mpf(1))
            -iv_pow(iv.mpf(2), -S_ip1-iv.mpf(2))/(S_ip1+iv.mpf(3))
            -iv.mpf(2)*iv.log(iv.mpf(3))
            +iv.mpf(13)/iv.mpf(6)
        )
        interval_list.append(val)
        
        U = val.b
        if worst is None or U>worst:
            worst=U
        
    return interval_list, worst

F_second_at_half_result, F_second_at_half_largest = check_F_second_at_half_negative(s_i_list,step)
print(F_second_at_half_largest)

[-0.056589005897916599963, -0.056589005897916599963]


In [87]:
def check_F_at_zero_positive(s_i_list, step, a=0.72, c=0.5):
    # Check for F_s(0)>0.
    
    interval_list = []
    worst = None
    
    A=iv.mpf(a)
    C=iv.mpf(c)
    for s_i in s_i_list:
        S_i = iv.mpf(s_i)
        S_ip1 = iv.mpf(s_i+step)
        b_lower = (
            iv_pow(C, iv.mpf(2)*S_ip1)/S_ip1
            -iv.mpf(2)*iv_pow(C, S_i+iv.mpf(1))/(S_i+iv.mpf(1))
            -iv_pow(C, S_i+iv.mpf(3))/(S_i+iv.mpf(3))
            -iv.mpf(2)*iv_atanh(C)
            +(iv.mpf(2)/iv.mpf(3))*iv_pow(C, iv.mpf(3))
            +iv.mpf(2)*C
        )
        val = (
            iv.mpf(2)*A
            -iv.mpf(2)*iv_pow(A,S_i)/S_i
            +beta_iv(S_ip1)/iv.mpf(2)
            +b_lower
        )
        interval_list.append(val)
        
        L = val.a
        if worst is None or L<worst:
            worst=L
        
    return interval_list, worst

F_at_zero_result, F_at_zero_smallest = check_F_at_zero_positive(s_i_list,step)
print(F_at_zero_smallest)


[0.097021343352847289054, 0.097021343352847289054]


In [88]:
def check_F_at_half_positive(s_i_list, step, a=0.72, c=0.5):
    # Check for F_s(1/2)>0.
    
    interval_list = []
    worst = None
    
    A=iv.mpf(a)
    C=iv.mpf(c)
    for s_i in s_i_list:
        S_i = iv.mpf(s_i)
        S_ip1 = iv.mpf(s_i+step)
        b_lower = (
            iv_pow(C, iv.mpf(2)*S_ip1)/S_ip1
            -iv.mpf(2)*iv_pow(C, S_i+iv.mpf(1))/(S_i+iv.mpf(1))
            -iv_pow(C, S_i+iv.mpf(3))/(S_i+iv.mpf(3))
            -iv.mpf(2)*iv_atanh(C)
            +(iv.mpf(2)/iv.mpf(3))*iv_pow(C, iv.mpf(3))
            +iv.mpf(2)*C
        )
        val = (
            iv.mpf(1)/(iv.mpf(2)*S_ip1)
            -iv_pow(iv.mpf(2), iv.mpf(1)-iv.mpf(1)/S_ip1)*S_ip1/(S_i+iv.mpf(1))
            +iv.mpf(2)*A
            -iv.mpf(2)*iv_pow(A,S_i)/S_i
            +beta_iv(S_ip1)/iv.mpf(4)
            +(iv.mpf(3)/iv.mpf(4))*b_lower
        )
        interval_list.append(val)
        
        L = val.a
        if worst is None or L<worst:
            worst=L
        
    return interval_list, worst

F_at_half_result, F_at_half_smallest = check_F_at_half_positive(s_i_list,step)
print(F_at_half_smallest)


[0.023166684187571839892, 0.023166684187571839892]
